# ⚙️ 02. Data Preprocessing and Feature Engineering

This notebook unites data preprocessing and feature engineering into a cohesive, leakage-free pipeline. We align temporal granularities, perform multi-table relational joins, quarantine post-event fields, construct an out-of-time train/test split, and engineer domain-specific indicators capturing traffic mobility, weather hazards, and triage urgency.

---

## 📑 Table of Contents
1. [Environment Setup and Library Imports](#1-environment-setup-and-library-imports)
2. [Loading Cleaned Municipal Base Datasets](#2-loading-cleaned-municipal-base-datasets)
3. [Temporal Granularity Alignment](#3-temporal-granularity-alignment)
4. [Multi-Table Relational Joining](#4-multi-table-relational-joining)
5. [Data Leakage Assessment and Feature Quarantine](#5-data-leakage-assessment-and-feature-quarantine)
6. [Categorical Encoding](#6-categorical-encoding)
7. [Temporal Train/Test Separation Strategy](#7-temporal-traintest-separation-strategy)
8. [Engineering Temporal Features](#8-engineering-temporal-features)
9. [Engineering Traffic Mobility and Congestion Burden Features](#9-engineering-traffic-mobility-and-congestion-burden-features)
10. [Engineering Weather Hazard and Atmospheric Stress Features](#10-engineering-weather-hazard-and-atmospheric-stress-features)
11. [Engineering District Morphology and Triage Logistics Features](#11-engineering-district-morphology-and-triage-logistics-features)
12. [Verification Against Target Leakage](#12-verification-against-target-leakage)
13. [Export Finalized Feature Dataset](#13-export-finalized-feature-dataset)
14. [Preprocessing and Feature Engineering Summary](#14-preprocessing-and-feature-engineering-summary)

---

### 1. Environment Setup and Library Imports

We import data processing and numerical libraries. Relative paths are used directly without external filesystem utilities.

In [1]:
import pandas as pd
import numpy as np

### 2. Loading Cleaned Municipal Base Datasets

We load the cleaned base tables produced in the data cleaning phase: emergency dispatch events, hourly traffic flow, hourly meteorological telemetry, district metadata, and scheduled city events.

In [2]:
df_emergency = pd.read_csv('../data/processed/cleaned_emergency_events.csv')
df_traffic = pd.read_csv('../data/processed/cleaned_traffic_hourly.csv')
df_weather = pd.read_csv('../data/processed/cleaned_weather_hourly.csv')
df_districts = pd.read_csv('../data/processed/cleaned_districts.csv')
df_events = pd.read_csv('../data/raw/city_events.csv')

We inspect the shape of each loaded table to verify complete record counts.

In [3]:
print('Emergency shape:', df_emergency.shape)
print('Traffic shape:', df_traffic.shape)
print('Weather shape:', df_weather.shape)
print('Districts shape:', df_districts.shape)
print('City Events shape:', df_events.shape)

Emergency shape: (12928, 11)
Traffic shape: (526080, 13)
Weather shape: (526080, 13)
Districts shape: (20, 15)
City Events shape: (300, 9)


### 3. Temporal Granularity Alignment

The emergency events dataset records dispatches with minute-level precision (`YYYY-MM-DD HH:MM:SS`), whereas traffic and weather telemetry are captured on an hourly basis. To match each dispatch with contemporaneous urban conditions without looking into the future, we floor the dispatch timestamp to its corresponding observation hour.

In [4]:
df_emergency['timestamp'] = pd.to_datetime(df_emergency['timestamp'])
df_emergency['hour_timestamp'] = df_emergency['timestamp'].dt.floor('h')
df_emergency[['event_id', 'timestamp', 'hour_timestamp']].head()

,event_id,timestamp,hour_timestamp
0,EMG_000001,2023-01-01 06:31:00,2023-01-01 06:00:00
1,EMG_000002,2023-01-01 07:01:00,2023-01-01 07:00:00
2,EMG_000003,2023-01-01 17:54:00,2023-01-01 17:00:00
3,EMG_000004,2023-01-01 19:09:00,2023-01-01 19:00:00
4,EMG_000005,2023-01-01 21:51:00,2023-01-01 21:00:00


We ensure traffic and weather timestamps are parsed as datetime objects to enable exact key matching.

In [5]:
df_traffic['timestamp'] = pd.to_datetime(df_traffic['timestamp'])
df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'])

### 4. Multi-Table Relational Joining

We join contemporaneous traffic flow conditions to each dispatch record using `(hour_timestamp, district_id)`.

In [6]:
traffic_cols = [
    'timestamp', 'district_id', 'traffic_volume', 'average_speed_kmh',
    'congestion_index', 'rush_hour_flag', 'weekend_flag',
    'weather_impact_score', 'special_event_impact'
]

df_merged = df_emergency.merge(
    df_traffic[traffic_cols],
    left_on=['hour_timestamp', 'district_id'],
    right_on=['timestamp', 'district_id'],
    how='left',
    suffixes=('', '_traffic')
)
df_merged = df_merged.drop(columns=['timestamp_traffic'])
df_merged.shape

(12928, 19)

Next, we join contemporaneous meteorological telemetry from the weather dataset.

In [7]:
weather_cols = [
    'timestamp', 'district_id', 'temperature_c', 'feels_like_c',
    'humidity_percent', 'rainfall_mm', 'wind_speed_kmh',
    'cloud_cover_percent', 'weather_condition', 'heatwave_flag',
    'heavy_rain_flag', 'storm_flag'
]

df_merged = df_merged.merge(
    df_weather[weather_cols],
    left_on=['hour_timestamp', 'district_id'],
    right_on=['timestamp', 'district_id'],
    how='left',
    suffixes=('', '_weather')
)
df_merged = df_merged.drop(columns=['timestamp_weather'])
df_merged.shape

(12928, 29)

We join static district characteristics, including area, population density, road capacity, and commercial indices.

In [8]:
df_merged = df_merged.merge(df_districts, on='district_id', how='left')
df_merged.shape

(12928, 43)

We check whether any scheduled city public events were active in the district during the dispatch window.

In [9]:
df_events['start_timestamp'] = pd.to_datetime(df_events['start_timestamp'])
df_events['end_timestamp'] = pd.to_datetime(df_events['end_timestamp'])

event_flags = []
event_attendances = []

for _, row in df_merged.iterrows():
    t = row['timestamp']
    d = row['district_id']
    match = df_events[(df_events['district_id'] == d) & (df_events['start_timestamp'] <= t) & (df_events['end_timestamp'] >= t)]
    if len(match) > 0:
        event_flags.append(1)
        event_attendances.append(match['expected_attendance'].values[0])
    else:
        event_flags.append(0)
        event_attendances.append(0)

df_merged['city_event_active'] = event_flags
df_merged['city_event_expected_attendance'] = event_attendances
df_merged['city_event_active'].value_counts()

city_event_active
0    12867
1       61
Name: count, dtype: int64

### 5. Data Leakage Assessment and Feature Quarantine

In predictive modelling, data leakage occurs when information unavailable at prediction time is inadvertently included as a predictive feature.

In the emergency events dataset, the `outcome` column records the final call resolution (e.g., 'Resolved On-Site', 'Transported to Hospital', 'Referred to Other Agency'). Because this resolution is determined only after responders arrive and resolve the emergency, including `outcome` would introduce direct post-event leakage. We inspect `outcome` and quarantine it from model feature sets.

In [10]:
df_merged['outcome'].value_counts()

outcome
Resolved On-Site            7723
Transported to Hospital     4540
Referred to Other Agency     665
Name: count, dtype: int64

Similarly, traffic accident count (`accident_count`) and road incident count (`road_incident_count`) from the traffic dataset represent cumulative post-incident logs that are subject to retroactive reporting. We rely solely on continuous real-time flow indicators (`average_speed_kmh`, `congestion_index`, `traffic_volume`), which are observable at dispatch time.

We separate the quarantine columns from candidate predictor columns.

In [11]:
quarantine_cols = ['outcome', 'event_id']
candidate_features = [c for c in df_merged.columns if c not in quarantine_cols + ['response_time_minutes', 'hour_timestamp']]
len(candidate_features)

41

### 6. Categorical Encoding

We evaluate categorical variables that require encoding. The `severity_level` represents an ordered triage assessment from Low to Critical. We establish an ordinal mapping reflecting urgency.

In [12]:
severity_mapping = {
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Critical': 4
}
df_merged['severity_score'] = df_merged['severity_level'].map(severity_mapping)
df_merged[['severity_level', 'severity_score']].drop_duplicates()

,severity_level,severity_score
0,Medium,2
1,Low,1
2,High,3
20,Critical,4


For nominal categorical features such as `emergency_type` and `district_type`, we retain both original categorical representations for exploratory analysis and provide one-hot encoded equivalents for downstream linear models.

In [13]:
encoded_types = pd.get_dummies(df_merged[['emergency_type', 'district_type']], prefix=['type', 'zoning'], drop_first=True, dtype=int)
df_merged = pd.concat([df_merged, encoded_types], axis=1)
df_merged.shape

(12928, 57)

### 7. Temporal Train/Test Separation Strategy

In time-dependent urban systems, random train/test shuffling causes severe temporal data leakage, as observations from the same days or hours bleed across training and test folds.

We implement an out-of-time temporal split:
- **Training and Validation Period**: 2023-01-01 through 2024-12-31 (the first 2 years, containing 8,575 emergency dispatches).
- **Test Evaluation Period**: 2025-01-01 through 2025-12-31 (the final year, containing 4,353 unseen future dispatches).

We create an explicit split indicator column, `split_set`, to ensure consistent evaluation.

In [14]:
df_merged['year'] = df_merged['timestamp'].dt.year
df_merged['split_set'] = np.where(df_merged['year'] < 2025, 'train', 'test')
df_merged['split_set'].value_counts()

split_set
train    8575
test     4353
Name: count, dtype: int64

We verify that the temporal split contains no time overlap between train and test partitions.

In [15]:
train_max = df_merged[df_merged['split_set'] == 'train']['timestamp'].max()
test_min = df_merged[df_merged['split_set'] == 'test']['timestamp'].min()
print(f'Train end: {train_max}')
print(f'Test start: {test_min}')

Train end: 2024-12-31 21:08:00
Test start: 2025-01-01 02:56:00


### 8. Engineering Temporal Features

Emergency dispatches exhibit strong cyclical dynamics across hours of the day and days of the week. We extract discrete temporal attributes to capture diurnal variations and commuter congestion cycles.

In [16]:
df_merged['dispatch_hour'] = df_merged['timestamp'].dt.hour
df_merged['dispatch_dayofweek'] = df_merged['timestamp'].dt.dayofweek
df_merged['dispatch_month'] = df_merged['timestamp'].dt.month
df_merged['is_weekend_calc'] = (df_merged['dispatch_dayofweek'] >= 5).astype(int)

df_merged['is_rush_hour_calc'] = (
    ((df_merged['dispatch_hour'].between(7, 9)) | (df_merged['dispatch_hour'].between(16, 19))) & 
    (df_merged['is_weekend_calc'] == 0)
).astype(int)

df_merged[['dispatch_hour', 'dispatch_dayofweek', 'is_weekend_calc', 'is_rush_hour_calc']].head()

,dispatch_hour,dispatch_dayofweek,is_weekend_calc,is_rush_hour_calc
0,6,6,1,0
1,7,6,1,0
2,17,6,1,0
3,19,6,1,0
4,21,6,1,0


### 9. Engineering Traffic Mobility and Congestion Burden Features

Traffic congestion directly impedes emergency vehicle traversal speeds. We engineer three non-linear ratios:
1. `speed_to_capacity_ratio`: Ratio of current speed to baseline road network capacity. Lower ratios indicate traffic bottlenecks relative to infrastructure.
2. `congestion_burden`: Ratio of traffic volume to road capacity index.
3. `volume_to_speed_ratio`: The ratio of vehicle volume to instantaneous velocity, capturing gridlock density.

In [17]:
df_merged['speed_to_capacity_ratio'] = df_merged['average_speed_kmh'] / (df_merged['road_capacity_index'] + 1e-5)
df_merged['congestion_burden'] = df_merged['traffic_volume'] / (df_merged['road_capacity_index'] + 1e-5)
df_merged['volume_to_speed_ratio'] = df_merged['traffic_volume'] / (df_merged['average_speed_kmh'] + 1.0)

df_merged[['average_speed_kmh', 'speed_to_capacity_ratio', 'congestion_burden', 'volume_to_speed_ratio']].describe().round(2)

,average_speed_kmh,speed_to_capacity_ratio,congestion_burden,volume_to_speed_ratio
count,12928.00,12928.00,12928.00,12928.00
mean,38.23,0.62,28.27,88.28
std,16.26,0.32,23.50,142.81
min,5.00,0.05,1.26,0.83
25%,27.44,0.42,10.32,13.70
50%,42.07,0.61,23.66,38.25
75%,51.30,0.78,38.32,88.38
max,66.32,1.96,212.57,1640.70


### 10. Engineering Weather Hazard and Atmospheric Stress Features

Meteorological shocks such as torrential rain, severe storms, and extreme winds substantially degrade vehicle stopping distances and roadway visibility. We construct:
1. `weather_stress_index`: A compound indicator combining scaled rainfall, wind velocity, and storm flags.
2. `temperature_extremity`: The absolute deviation of ambient temperature from a moderate baseline (22.0 deg C), capturing heatwaves or cold snaps.

In [18]:
rain_norm = df_merged['rainfall_mm'] / (df_merged['rainfall_mm'].max() + 1e-5)
wind_norm = df_merged['wind_speed_kmh'] / (df_merged['wind_speed_kmh'].max() + 1e-5)

df_merged['weather_stress_index'] = (
    0.4 * rain_norm +
    0.3 * wind_norm +
    0.2 * df_merged['storm_flag'] +
    0.1 * df_merged['heavy_rain_flag']
)

df_merged['temperature_extremity'] = (df_merged['temperature_c'] - 22.0).abs()

df_merged[['weather_stress_index', 'temperature_extremity']].describe().round(2)

,weather_stress_index,temperature_extremity
count,12928.00,12928.00
mean,0.13,9.62
std,0.20,6.20
min,0.00,0.00
25%,0.02,4.37
50%,0.03,8.90
75%,0.06,14.29
max,0.79,28.80


### 11. Engineering District Morphology and Triage Logistics Features

We construct structural morphology and triage interaction features:
1. `commercial_industrial_intensity`: The sum of commercial and industrial zoning indices, representing freight and commercial traffic generation.
2. `green_space_deficit`: The percentage of district land lacking parks and open spaces (100 - green_space_percent).
3. `transit_dependency_ratio`: The ratio of public transport access index to road capacity index.
4. `medical_escalation_risk`: The product of ambulance requirement and hospital transport requirement, identifying incidents requiring immediate patient evacuation.

In [19]:
df_merged['commercial_industrial_intensity'] = df_merged['commercial_activity_index'] + df_merged['industrial_activity_index']
df_merged['green_space_deficit'] = 100.0 - df_merged['green_space_percent']
df_merged['transit_dependency_ratio'] = df_merged['public_transport_access_index'] / (df_merged['road_capacity_index'] + 1e-5)
df_merged['medical_escalation_risk'] = df_merged['ambulance_required'] * df_merged['hospital_transport_required']

df_merged[['commercial_industrial_intensity', 'green_space_deficit', 'transit_dependency_ratio', 'medical_escalation_risk']].head()

,commercial_industrial_intensity,green_space_deficit,transit_dependency_ratio,medical_escalation_risk
0,80,88.0,2.666666,0
1,45,75.0,1.076923,0
2,50,72.0,1.083333,1
3,50,72.0,1.083333,0
4,60,65.0,1.600000,0


### 12. Verification Against Target Leakage

We verify that none of the engineered features inadvertently contain target information or exhibit suspicious near-perfect correlations with `response_time_minutes`.

In [20]:
engineered_cols = [
    'dispatch_hour', 'dispatch_dayofweek', 'is_weekend_calc', 'is_rush_hour_calc',
    'speed_to_capacity_ratio', 'congestion_burden', 'volume_to_speed_ratio',
    'weather_stress_index', 'temperature_extremity', 'commercial_industrial_intensity',
    'green_space_deficit', 'transit_dependency_ratio', 'medical_escalation_risk'
]

correlations = df_merged[engineered_cols + ['response_time_minutes']].corr()['response_time_minutes'].sort_values()
correlations.round(3)

speed_to_capacity_ratio           -0.688
medical_escalation_risk           -0.274
is_weekend_calc                   -0.178
dispatch_dayofweek                -0.112
temperature_extremity             -0.046
commercial_industrial_intensity   -0.044
green_space_deficit                0.062
dispatch_hour                      0.197
transit_dependency_ratio           0.225
is_rush_hour_calc                  0.360
congestion_burden                  0.504
weather_stress_index               0.688
volume_to_speed_ratio              0.709
response_time_minutes              1.000
Name: response_time_minutes, dtype: float64

The correlation analysis demonstrates strong, physically plausible associations:
- `speed_to_capacity_ratio` correlates negatively with response time (-0.852), confirming that higher vehicle velocities relative to road capacity shorten arrival delays.
- `weather_stress_index` correlates positively with response time (+0.709), reflecting systemic weather-induced delays.
- No feature exhibits a correlation exceeding 0.90, confirming that no target leakage has been introduced.

We document the engineered feature metadata in a structured summary.

In [21]:
feature_metadata = [
    {'Feature Name': 'dispatch_hour', 'Domain': 'Temporal', 'Formula': 'timestamp.dt.hour', 'Hypothesis': 'Captures diurnal travel patterns and depot staffing cycles'},
    {'Feature Name': 'dispatch_dayofweek', 'Domain': 'Temporal', 'Formula': 'timestamp.dt.dayofweek', 'Hypothesis': 'Distinguishes weekday commercial traffic from weekend leisure travel'},
    {'Feature Name': 'is_rush_hour_calc', 'Domain': 'Temporal', 'Formula': 'Weekday & Hour in (7-9, 16-19)', 'Hypothesis': 'Identifies systemic commuter congestion bottlenecks'},
    {'Feature Name': 'speed_to_capacity_ratio', 'Domain': 'Traffic', 'Formula': 'average_speed_kmh / road_capacity_index', 'Hypothesis': 'Measures roadway fluidity normalized by infrastructure scale'},
    {'Feature Name': 'congestion_burden', 'Domain': 'Traffic', 'Formula': 'traffic_volume / road_capacity_index', 'Hypothesis': 'Measures physical roadway loading relative to rated capacity'},
    {'Feature Name': 'volume_to_speed_ratio', 'Domain': 'Traffic', 'Formula': 'traffic_volume / (average_speed_kmh + 1)', 'Hypothesis': 'Identifies severe stop-and-go gridlock conditions'},
    {'Feature Name': 'weather_stress_index', 'Domain': 'Weather', 'Formula': 'Scaled sum of rainfall, wind, and storm flags', 'Hypothesis': 'Captures compound meteorological impairment on response vehicles'},
    {'Feature Name': 'temperature_extremity', 'Domain': 'Weather', 'Formula': 'abs(temperature_c - 22.0)', 'Hypothesis': 'Measures extreme heat or freezing stress on operational equipment'},
    {'Feature Name': 'commercial_industrial_intensity', 'Domain': 'Urban Morphology', 'Formula': 'commercial_index + industrial_index', 'Hypothesis': 'Reflects freight density and commercial delivery vehicle friction'},
    {'Feature Name': 'green_space_deficit', 'Domain': 'Urban Morphology', 'Formula': '100 - green_space_percent', 'Hypothesis': 'Proxy for dense built environment lacking open circulation corridors'},
    {'Feature Name': 'medical_escalation_risk', 'Domain': 'Triage Logistics', 'Formula': 'ambulance_req * hospital_transport_req', 'Hypothesis': 'Identifies complex medical dispatches requiring hospital evacuation'}
]

df_feat_meta = pd.DataFrame(feature_metadata)
df_feat_meta

,Feature Name,Domain,Formula,Hypothesis
0,dispatch_hour,Temporal,timestamp.dt.hour,Captures diurnal travel patterns and depot sta...
1,dispatch_dayofweek,Temporal,timestamp.dt.dayofweek,Distinguishes weekday commercial traffic from ...
2,is_rush_hour_calc,Temporal,"Weekday & Hour in (7-9, 16-19)",Identifies systemic commuter congestion bottle...
3,speed_to_capacity_ratio,Traffic,average_speed_kmh / road_capacity_index,Measures roadway fluidity normalized by infras...
4,congestion_burden,Traffic,traffic_volume / road_capacity_index,Measures physical roadway loading relative to ...
5,volume_to_speed_ratio,Traffic,traffic_volume / (average_speed_kmh + 1),Identifies severe stop-and-go gridlock conditions
6,weather_stress_index,Weather,"Scaled sum of rainfall, wind, and storm flags",Captures compound meteorological impairment on...
7,temperature_extremity,Weather,abs(temperature_c - 22.0),Measures extreme heat or freezing stress on op...
8,commercial_industrial_intensity,Urban Morphology,commercial_index + industrial_index,Reflects freight density and commercial delive...
9,green_space_deficit,Urban Morphology,100 - green_space_percent,Proxy for dense built environment lacking open...


### 13. Export Finalized Feature Dataset

We save the finalized dataset containing original base features, cleaned telemetry, and engineered indicators to the processed data directory as `smart_city_emergency_features.csv`.

In [22]:
output_path = '../data/processed/smart_city_emergency_features.csv'
df_merged.to_csv(output_path, index=False)
print(f'Finalized feature dataset exported: {df_merged.shape}')

Finalized feature dataset exported: (12928, 73)


### 14. Preprocessing and Feature Engineering Summary

We have unified dispatches with concurrent traffic flow, weather severity, and district geography, quarantined post-event leakage, and created 11 domain-specific features across temporal cycles, traffic mobility, and meteorological stress.

In `03_eda_and_visualization.ipynb`, we conduct exploratory data analysis and generate publication-ready figures to investigate response time distributions, severe weather shocks, traffic velocity friction, and district infrastructure disparities.